# Dry-run env validator (Colab)

Validates the full training pipeline (load → LoRA → SFTTrainer → save) end-to-end with a **known-working pinned dep set**, on a tiny model + synthetic data. Total runtime: ~3 min.

**Why this exists:** previous RunPod session burned time debugging dep mismatches. From now on, this notebook is run *first* on Colab (free) before any RunPod session. If it works here, the same pins should work on RunPod.

**How to use:** Runtime → A100 (or T4 — model is tiny, runtime doesn't matter much), Run all. The last cell prints a clean `pip freeze` to copy into `requirements.txt`.

**Note on model size:** uses Qwen2.5-Coder-0.5B for speed — same architecture family as 7B, so dep stack validation transfers.

In [ ]:
# Install pinned deps. Reasoning for each pin:
#  - trl<0.13: keeps DataCollatorForCompletionOnlyLM exported at top level
#  - transformers<4.50: avoids string-annotation issues in custom_op
#  - peft<0.14: avoids torch.distributed.tensor reference issues
#  - accelerate / datasets / wandb: compatible ranges
# torch is whatever Colab gives us (don't fight the runtime)
!pip install -q \
    'transformers>=4.45,<4.50' \
    'trl>=0.11,<0.13' \
    'peft>=0.13,<0.14' \
    'accelerate>=1.0,<2.0' \
    'datasets>=3.0,<4.0' \
    'wandb>=0.18,<1.0' \
    'cairosvg>=2.7'
print('install done')

In [ ]:
# Imports — fail fast if any pin is wrong
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset

print(f'torch:        {torch.__version__}  (cuda={torch.cuda.is_available()})')
import transformers, peft, trl, accelerate, datasets
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'trl:          {trl.__version__}')
print(f'accelerate:   {accelerate.__version__}')
print(f'datasets:     {datasets.__version__}')

In [ ]:
# Load tiny base model (0.5B for speed; same arch family as 7B)
MODEL = 'Qwen/Qwen2.5-Coder-0.5B'
print(f'Loading {MODEL}...')
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
print(f'Loaded. Device: {model.device}, dtype: {model.dtype}')

In [ ]:
# Wrap with LoRA — same config as our real training (r=32, alpha=64)
lora_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Synthetic dataset — exercises data collator + format separator without
# needing to upload the real parquets. Same prompt format as production.
INSTRUCTION = 'Generate an SVG glyph for: '
RESPONSE_TPL = '\nSVG:\n'
FAKE_DATA = [
    {"text": f"{INSTRUCTION}letter A in sans-serif{RESPONSE_TPL}<svg viewBox='0 0 100 100'><path d='M50 0L0 100 100 100 Z'/></svg>"},
    {"text": f"{INSTRUCTION}letter B in serif{RESPONSE_TPL}<svg viewBox='0 0 100 100'><path d='M10 10 L90 50 L10 90 Z'/></svg>"},
    {"text": f"{INSTRUCTION}letter C in display{RESPONSE_TPL}<svg viewBox='0 0 100 100'><path d='M50 50 m-30 0 a30 30 0 1 0 60 0'/></svg>"},
    {"text": f"{INSTRUCTION}letter D in handwriting{RESPONSE_TPL}<svg viewBox='0 0 100 100'><path d='M20 20 Q 50 10 80 50'/></svg>"},
    {"text": f"{INSTRUCTION}letter E in monospace{RESPONSE_TPL}<svg viewBox='0 0 100 100'><path d='M10 10 L90 10 M10 50 L70 50 M10 90 L90 90'/></svg>"},
] * 4   # 20 examples total
train_ds = Dataset.from_list(FAKE_DATA)
val_ds = Dataset.from_list(FAKE_DATA[:5])
print(f'train: {len(train_ds)}, val: {len(val_ds)}')

In [ ]:
# Train 5 steps with the real config knobs
sft_args = SFTConfig(
    output_dir='/content/dryrun_out',
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_seq_length=256,
    logging_steps=1,
    save_steps=100,
    eval_strategy='no',
    push_to_hub=False,
    report_to='none',
    dataset_text_field='text',
    max_steps=5,
    bf16=torch.cuda.is_available(),
)
collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TPL,
    tokenizer=tok,
)
trainer = SFTTrainer(
    model=model, args=sft_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, tokenizer=tok,
)
trainer.train()
trainer.save_model('/content/dryrun_out/final')
print('\n✅ Dry-run complete. The pipeline wires up correctly.')

In [ ]:
# Capture the exact working environment
import subprocess
out = subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout
with open('/content/working_pip_freeze.txt', 'w') as f:
    f.write(out)
print('Wrote /content/working_pip_freeze.txt — download this and we\'ll lock in the deps.')
# Print just the key deps so we can paste them directly
for line in out.splitlines():
    if any(x in line.lower() for x in ['torch', 'transformers', 'peft', 'trl', 'accelerate', 'datasets', 'wandb']):
        print(line)